# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Scepter70/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Scepter70/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

import pandas as pd

fact = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet")
dim_content = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/dim_content.parquet")
dim_clients = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet")
print("Loaded. fact rows:", len(fact), "| dim_content rows:", len(dim_content))


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Run the code cell below first — it checks two signals. Paste the output to Claude before finalizing this text.**

Planned signals:
1. **Staleness** (`days_since_optimized`, from `last_optimized_date`) — the signal behind FlyRank's real refresh flags.
2. **Search volume** (`search_volume`, from `dim_content`) — the signal behind quick-win logic (high-volume, poorly-ranked pages are worth prioritizing).

*(Verdicts and final rule wording go here once the signal check below has run.)*

In [ ]:
import numpy as np

# One row per page for this month
page = fact.groupby(["client_hash_id", "content_hash_id"], as_index=False).agg(
    gsc_impressions=("gsc_impressions", "sum"),
    gsc_clicks=("gsc_clicks", "sum"),
    gsc_avg_position=("gsc_avg_position", "mean"),
)
page["ctr"] = np.where(page["gsc_impressions"] > 0, page["gsc_clicks"] / page["gsc_impressions"], np.nan)

page = page.merge(
    dim_content[["client_hash_id", "content_hash_id", "last_optimized_date", "search_volume", "is_deleted"]],
    on=["client_hash_id", "content_hash_id"], how="left"
)
page = page[page["is_deleted"] == False].copy()

page["last_optimized_date"] = pd.to_datetime(page["last_optimized_date"], errors="coerce")
page["days_since_optimized"] = (pd.Timestamp("2026-03-31") - page["last_optimized_date"]).dt.days

# --- SIGNAL 1: staleness vs CTR (staleness is the signal behind FlyRank's refresh flags) ---
page["staleness_bucket"] = pd.cut(
    page["days_since_optimized"],
    bins=[-1, 30, 90, 180, 100000],
    labels=["0-30d", "31-90d", "91-180d", "180d+"]
)
signal1 = page.groupby("staleness_bucket", observed=True).agg(
    n=("ctr", "size"), avg_ctr=("ctr", "mean")
).reset_index()
print("SIGNAL 1 - staleness vs CTR:")
print(signal1)
corr1 = page[["days_since_optimized", "ctr"]].dropna().corr().iloc[0, 1]
print(f"Correlation(days_since_optimized, ctr) = {corr1:.4f}")

# --- SIGNAL 2: search_volume vs position (volume is the signal behind quick-win logic) ---
page["volume_bucket"] = pd.qcut(page["search_volume"].rank(method="first"), 4,
    labels=["Q1 low", "Q2", "Q3", "Q4 high"])
signal2 = page.groupby("volume_bucket", observed=True).agg(
    n=("gsc_avg_position", "size"), avg_position=("gsc_avg_position", "mean"),
    avg_impressions=("gsc_impressions", "mean")
).reset_index()
print("\nSIGNAL 2 - search_volume vs position/impressions:")
print(signal2)
corr2 = page[["search_volume", "gsc_avg_position"]].dropna().corr().iloc[0, 1]
print(f"Correlation(search_volume, gsc_avg_position) = {corr2:.4f}")


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.